## Train GPT on complete_mb.txt using GPT2 Tokenizer

**Dataset:** `main/resources/complete_mb.txt` (~11.7 MB, ~2.9M tokens after GPT2 tokenization)

**Config choices:**
| Param | Value | Reason |
|---|---|---|
| vocab_size | 50257 | GPT2 tiktoken |
| context_len | 512 | 2x char model; good for prose |
| emb_dim | 384 | Same as char model |
| n_heads | 6 | head_dim = 64 (384/6) |
| n_layers | 6 | ~30M total params |
| batch_size | 32 | Safe for MPS with context_len=512 |
| dropout | 0.1 | Lower since dataset is smaller (~2.9M tokens) |
| max_train_iters | 5000 | ~80M tokens seen (27x through data) |

**Estimated model size:** ~30M params (~120MB in float32, ~60MB in bfloat16)

In [1]:
import sys
import os

# __file__ is not defined in Jupyter — use os.path.abspath('') to get the
# notebook's working directory (works when opened from its own folder)
sys.path.insert(0, os.path.abspath(''))

import torch
import tiktoken
from gpt_model import GPTModel2EX1, GPTConfig
from train_gpt import TrainGPTModel, TrainConfig

device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: mps


In [2]:
# Load and tokenize all books with GPT2 tokenizer
# <|endoftext|> (token 50256) is inserted between books so the model learns
# they are separate documents and doesn't connect the ending of one to the start of the next.
PROJECT_ROOT = "/Users/ashritkuma.samudrala/lnex/ex_llm_rag"
BOOKS_DIR    = f"{PROJECT_ROOT}/main/resources/books"

book_files = [
    "complete_mb.txt",
    "tinyshakespeare.txt",
    "val_ramn.txt",
]

enc = tiktoken.get_encoding('gpt2')
EOT = enc.eot_token  # 50256

train_tokens = []
val_tokens   = []

for fname in book_files:
    path = f"{BOOKS_DIR}/{fname}"
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    tokens = enc.encode(text)

    # Per-book 90/10 split — val set represents all books, not just the last one
    split = int(0.9 * len(tokens))
    train_tokens.extend(tokens[:split] + [EOT])
    val_tokens.extend(tokens[split:] + [EOT])

    print(f"{fname:30s}: {len(tokens):>10,} tokens  train={split:,}  val={len(tokens)-split:,}")

data       = torch.tensor(train_tokens + val_tokens, dtype=torch.long)
train_data = torch.tensor(train_tokens, dtype=torch.long)
val_data   = torch.tensor(val_tokens,   dtype=torch.long)

print(f"\nTrain tokens : {len(train_data):,}")
print(f"Val tokens   : {len(val_data):,}")
print(f"Vocab size   : {enc.n_vocab}")

complete_mb.txt               :  3,144,549 tokens  train=2,830,094  val=314,455
tinyshakespeare.txt           :    338,025 tokens  train=304,222  val=33,803
val_ramn.txt                  :    109,117 tokens  train=98,205  val=10,912

Train tokens : 3,232,524
Val tokens   : 359,173
Vocab size   : 50257


In [3]:
# ── GPT Config ─────────────────────────────────────────────────────────────
# ~3.55M total tokens. Chinchilla optimal: ~177K params.
# Using ~5M params (emb_dim=256, 4 heads, 6 layers) — overparameterized but
# manageable with dropout=0.2. Much better than the 30M model (8x less overfitting risk).
gpt_cfg = GPTConfig()
gpt_cfg.vocab_size           = 50257
gpt_cfg.context_len          = 512
gpt_cfg.batch_size           = 32
gpt_cfg.emb_dim              = 256
gpt_cfg.n_heads              = 4     # head_dim = 64 (256/4)
gpt_cfg.n_transformer_blocks = 6
gpt_cfg.dropout              = 0.2   # higher than before — more regularization for small dataset

# ── Train Config ────────────────────────────────────────────────────────────
train_cfg = TrainConfig()
train_cfg.learning_rate      = 3e-4
train_cfg.min_lr             = 3e-5  # cosine decays to here
train_cfg.warmup_iters       = 100
train_cfg.max_train_iters    = 5000
train_cfg.loss_eval_interval = 500
train_cfg.loss_eval_itrs     = 10
train_cfg.context_len        = gpt_cfg.context_len
train_cfg.batch_size         = gpt_cfg.batch_size
train_cfg.use_autocast       = False  # bfloat16 autocast conflicts with torch.compile+no_grad on MPS
train_cfg.grad_clip          = 1.0
train_cfg.checkpoint_path    = f"{PROJECT_ROOT}/main/resources/models/books/books_5M_checkpoint.model"

print("GPT Config:")
print(f"  vocab_size={gpt_cfg.vocab_size}, context_len={gpt_cfg.context_len}, batch_size={gpt_cfg.batch_size}")
print(f"  emb_dim={gpt_cfg.emb_dim}, n_heads={gpt_cfg.n_heads}, n_layers={gpt_cfg.n_transformer_blocks}, dropout={gpt_cfg.dropout}")
print("\nTrain Config:")
print(f"  lr={train_cfg.learning_rate} → {train_cfg.min_lr} (cosine), warmup={train_cfg.warmup_iters}")
print(f"  iters={train_cfg.max_train_iters}, eval_interval={train_cfg.loss_eval_interval}, grad_clip={train_cfg.grad_clip}")

GPT Config:
  vocab_size=50257, context_len=512, batch_size=32
  emb_dim=256, n_heads=4, n_layers=6, dropout=0.2

Train Config:
  lr=0.0003 → 3e-05 (cosine), warmup=100
  iters=5000, eval_interval=500, grad_clip=1.0


In [4]:
# Instantiate model and print parameter count
model = GPTModel2EX1(gpt_cfg).to(device)

total_params = sum(p.numel() for p in model.parameters())
# Subtract lm_head params since it shares weights with wte (weight tying)
unique_params = total_params - model.lm_head.weight.numel()

print(f"Total params  : {total_params:,}")
print(f"Unique params : {unique_params:,}  (lm_head weight tied to wte)")
print(f"Model size    : ~{unique_params * 4 / 1e6:.1f} MB (float32)")

Total params  : 30,651,985
Unique params : 17,786,193  (lm_head weight tied to wte)
Model size    : ~71.1 MB (float32)


In [5]:
# torch.compile — fuses ops and reduces kernel launch overhead (~10-30% speedup)
# Requires PyTorch 2.0+. On MPS it works but may print a few warnings on first run.
# Falls back to uncompiled model if unsupported.
try:
    model = torch.compile(model)
    print("torch.compile enabled")
except Exception as e:
    print(f"torch.compile not available ({e}), running uncompiled")

torch.compile enabled


In [6]:
# Reset torch.compile cache — run this if you change model/training code mid-session
# without restarting the kernel, to force recompilation with the new code.
# Not needed after a kernel restart (in-memory cache is already cleared).
import torch
torch._dynamo.reset()
print("torch.compile cache reset")

torch.compile cache reset


In [14]:
# ── Benchmark: compile-only vs autocast-only vs baseline ───────────────────
# Tests both training iterations AND estimate_loss (which uses no_grad).
# Each config runs 20 train iters + 1 estimate_loss call.
# Iter 0 excluded from train avg (torch.compile warmup).
# torch.mps.synchronize() used for true GPU time.
import time

BENCH_ITERS = 20
EVAL_ITRS = 10  # small for quick benchmark

configs = {
    "baseline (no compile, no autocast)": {"compile": False, "autocast": False},
    "compile only":                        {"compile": True,  "autocast": False},
    "autocast only":                       {"compile": False, "autocast": True},
}

for label, cfg in configs.items():
    torch._dynamo.reset()
    bench_model = GPTModel2EX1(gpt_cfg).to(device)
    if cfg["compile"]:
        bench_model = torch.compile(bench_model)
    bench_model.train()

    bench_train_cfg = TrainConfig()
    bench_train_cfg.context_len    = gpt_cfg.context_len
    bench_train_cfg.batch_size     = gpt_cfg.batch_size
    bench_train_cfg.use_autocast   = cfg["autocast"]
    bench_train_cfg.loss_eval_itrs = EVAL_ITRS

    bench_trainer = TrainGPTModel(
        model=bench_model, config=bench_train_cfg,
        train_data=train_data, val_data=val_data, device=device
    )

    # ── Train iterations ──
    times = []
    for i in range(BENCH_ITERS):
        x, y = bench_trainer.get_batch()
        bench_trainer.optim.zero_grad()
        t0 = time.time()
        with bench_trainer._autocast_ctx():
            _, loss = bench_model(x, y)
        loss.backward()
        bench_trainer.optim.step()
        torch.mps.synchronize()
        times.append((time.time() - t0) * 1000)

    steady = times[1:]
    train_avg = sum(steady) / len(steady)

    # ── estimate_loss (uses no_grad — this is where compile+autocast breaks) ──
    eval_ok = True
    try:
        t0 = time.time()
        out = bench_trainer.estimate_loss()
        torch.mps.synchronize()
        eval_ms = (time.time() - t0) * 1000
    except RuntimeError as e:
        eval_ms = -1
        eval_ok = False

    eval_str = f"{eval_ms:7.0f}ms" if eval_ok else "  ERROR (dtype mismatch)"
    print(f"{label:40s}  train_avg={train_avg:7.0f}ms  eval={eval_str}")

    del bench_model, bench_trainer
    torch.mps.empty_cache()

print("\nPick the config with lowest train_avg where eval doesn't error.")

baseline (no compile, no autocast)        train_avg=   3427ms  eval=  12477ms
compile only                              train_avg=   1926ms  eval=  15904ms
autocast only                             train_avg=   1959ms  eval=  14539ms

Pick the config with lowest train_avg where eval doesn't error.


In [7]:
# Ensure model is in train mode — guards against running the generate cell before this
model.train()

trainer = TrainGPTModel(
    model      = model,
    config     = train_cfg,
    train_data = train_data,
    val_data   = val_data,
    device     = device
)

In [ ]:
# ── Resume training from checkpoint ────────────────────────────────────────
# Run this cell instead of the training cell above to continue from a saved checkpoint.
# Update max_train_iters in train_cfg before running to extend training.

CKPT_PATH = train_cfg.checkpoint_path

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
trainer.optim.load_state_dict(ckpt['optimizer_state_dict'])
start_iter = ckpt['iter'] + 1

print(f"Resumed from iter {ckpt['iter']}  val_loss={ckpt['val_loss']:.4f}")
print(f"Continuing from iter {start_iter} → {train_cfg.max_train_iters}")

model.train()
trainer.train_model(start_iter=start_iter)

In [43]:
train_cfg.learning_rate   = 5e-5   # lower max for the new cycle
train_cfg.min_lr          = 5e-5
train_cfg.warmup_iters    = 0     # shorter warmup
train_cfg.max_train_iters = 1
trainer.train_model()

Iter 0: Train Loss: 2.7960, Val Loss: 3.6667  lr: 5.00e-05  (eval: 14546ms)
  checkpoint saved → /Users/ashritkuma.samudrala/lnex/ex_llm_rag/main/resources/models/books/books_5M_checkpoint.model


In [ ]:
# Save final checkpoint
import os
SAVE_PATH = f"{PROJECT_ROOT}/main/resources/models/books/books_5M_5000iters.model"
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

out = trainer.estimate_loss()
torch.save({
    'iter'                : train_cfg.max_train_iters,
    'model_state_dict'    : model.state_dict(),
    'optimizer_state_dict': trainer.optim.state_dict(),
    'gpt_config'          : gpt_cfg,
    'train_config'        : train_cfg,
    'val_loss'            : out['val'],
}, SAVE_PATH)

print(f"Train Loss: {out['train']:.4f}  Val Loss: {out['val']:.4f}")
print(f"Saved → {SAVE_PATH}")

In [52]:
# Generate a sample
model.eval()

prompt = " "  # change to any prompt
prompt_tokens = enc.encode(prompt)
inp = torch.tensor([prompt_tokens], dtype=torch.long, device=device)

with torch.no_grad():
    out = model.generate(inp, max_len=100, temperature=1, eot_token=enc.eot_token)

generated = enc.decode(out[0].tolist())
print(generated)

  under Janamejaya, Bhaarata;
the mighty Ansumanas; others; and Maharathikas; Muhaschas; Kramohana;
Karushanohara; Saraholkata; Dhritasha; Shukritarajna; Damasvavaranochana;
Vasuyana; Niyama; Kushanadha; another meeting with Brihadradyumati;
Dhrishtha;
